<a href="https://colab.research.google.com/github/cyrus2281/notes/blob/main/Architecture/Architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Software Architecture Notes

Notes from [Software Architecture Monday](https://www.youtube.com/playlist?list=PLdsOZAx8I5umhnn5LLTNJbFgwA3xbycar) course by **Mark Richards**.

>[Software Architecture Notes](#scrollTo=PQ6Eg6AkHf12)

>[Event-Driven Architecture](#scrollTo=jdfzrk0oNW3C)

>>[Request/Reply Pattern](#scrollTo=TG1TXEEiVZEz)

>>>[Core Mechanism](#scrollTo=TG1TXEEiVZEz)

>>>[Implementation Techniques](#scrollTo=TG1TXEEiVZEz)

>>>>[Correlation IDs](#scrollTo=TG1TXEEiVZEz)

>>>>[Temporary Queues](#scrollTo=TG1TXEEiVZEz)

>>[How Apache Kafka Differs From Standard Messaging](#scrollTo=pS5Q0MhMaH5z)

>>>[Apache Kafka (Streaming Platform)](#scrollTo=pS5Q0MhMaH5z)

>>>[Standard Messaging (RabbitMQ, ActiveMQ, JMS)](#scrollTo=pS5Q0MhMaH5z)

>>>[Key Comparison Summary](#scrollTo=pS5Q0MhMaH5z)

>[Microservices](#scrollTo=QEFbv1JxEAVZ)

>>[Distributed Logging](#scrollTo=mPCFFXO7NexZ)

>>>[Log Consolidation](#scrollTo=mPCFFXO7NexZ)

>>>[Log Streaming](#scrollTo=mPCFFXO7NexZ)

>>>[Core Requirements for Distributed Logging](#scrollTo=mPCFFXO7NexZ)

>>>[Additional Industry Best Practices](#scrollTo=mPCFFXO7NexZ)

>[Soft Skills](#scrollTo=oJtSDgdCNUDT)

>>[The Knowledge Pyramid: Developers vs. Architects](#scrollTo=5W8N1uByAqzL)



# Event-Driven Architecture


## Request/Reply Pattern

The **Request/Reply pattern** allows for synchronous-like behavior ("pseudo-synchronous messaging") within an asynchronous event-driven architecture.

### Core Mechanism
*   **Structure:** Utilizes two distinct queues—a **Request Queue** (for sending) and a **Reply Queue** (for receiving).
*   **Workflow:**
    1.  Sender sends a message to the Request Queue.
    2.  Sender is free to perform other processing immediately after sending (asynchronous).
    3.  Sender performs a **blocking wait** on the Reply Queue when it requires the answer.
    4.  Receiver processes the request and sends the result to the Reply Queue.
    5.  Sender retrieves the response.

---

### Implementation Techniques

#### 1. Correlation IDs
This method is used when multiple responses sit in a shared Reply Queue. It ensures the sender retrieves only the response meant for its specific request.

*   **The Problem:** The Reply Queue may contain messages intended for other senders (e.g., IDs 120, 122).
*   **The Process:**
    1.  **Sender:** Sends a request with a unique **Message ID** (e.g., 124).
    2.  **Sender:** Waits on the Reply Queue using a **Message Selector** (or filter) looking for `CorrelationID == 124`.
    3.  **Receiver:** Gets the message, processes it, and sets the response's **Correlation ID** to match the original Message ID (124).
    4.  **Receiver:** Sends the message to the Reply Queue with a new unique Message ID (e.g., 857) but the matching Correlation ID.
    5.  **Sender:** Identifies the correct message via the Correlation ID and retrieves the data.

#### 2. Temporary Queues
A simpler alternative that does not use a shared reply queue initially.

*   **The Process:**
    1.  **Sender:** Sets a **"Reply To"** header in the message indicating a temporary queue (e.g., `TemporaryQueue T1`).
    2.  **Broker:** Creates this temporary queue; it is exclusive and unknown to others.
    3.  **Sender:** Performs a blocking wait on this specific temporary queue.
    4.  **Receiver:** Sends the response directly to the temporary queue specified in the header.
    5.  **Sender:** Receives the message (no selector/filter needed since the queue is private).
    6.  **Broker:** Removes the temporary queue once the interaction is complete.



## How Apache Kafka Differs From Standard Messaging

### Apache Kafka (Streaming Platform)
*   **Architecture Philosophy:** "Dumb Broker, Smart Consumer." The broker simply appends messages to a log; the consumer tracks its own position (offset).
*   **Data Type:** Good for **Operational Data** (metrics, logs, clickstreams, state changes).
*   **Payloads:** Optimized for small payloads (Key-Value pairs). Large payloads can degrade throughput significantly.
*   **Retention:** **Durable/Persistent by default.** Messages are retained for a configurable period (e.g., 7 days) or size, regardless of whether they have been consumed. This allows for "replayability."
*   **Throughput:** Extremely high (up to millions of messages/sec). Achieved through batching and sequential disk I/O.
*   **Topology:** Primarily **Publish/Subscribe** (Topics).
    *   Does not natively support complex routing logic (like message selectors or routing keys) inside the broker; this must be handled by the consumer or Kafka Streams.
*   **Scaling:** Horizontally scalable via **Partitioning**. Ordering is guaranteed only within a partition, not globally.

### Standard Messaging (RabbitMQ, ActiveMQ, JMS)
*   **Architecture Philosophy:** "Smart Broker, Dumb Consumer." The broker manages message state, delivery acknowledgments, and complex routing.
*   **Data Type:** Good for **Transactional Data** (orders, payments, user requests) where individual message guarantees are critical.
*   **Payloads:** Can handle larger payloads more gracefully than Kafka (though still limited by RAM/network).
*   **Retention:** **Transient by default.** Messages are typically deleted from the queue immediately after successful consumption (destructive read).
*   **Throughput:** Lower (approx. 4k–10k msgs/sec depending on persistence settings).
    *   *Note: While lower than Kafka, this is sufficient for most business transactional applications.*
*   **Topologies:** Supports complex, flexible topologies:
    *   **Point-to-Point (Queue):** Load balancing across consumers; message processed by only one consumer.
    *   **Publish/Subscribe (Topic):** Broadcast to all subscribers.
    *   **Request/Reply:** Natively supported via temporary queues and correlation IDs (as seen in Lesson 1).
*   **Routing:** sophisticated routing capabilities (e.g., Exchange types in RabbitMQ: Direct, Fanout, Topic, Headers) allow the broker to filter and route messages before they reach the consumer.

### Key Comparison Summary
| Feature | Kafka | Standard Messaging |
| :--- | :--- | :--- |
| **Message Lifecycle** | Log-based (persists until expiry) | Queue-based (deleted on consume) |
| **Consumer Complexity** | High (manages offsets) | Low (broker manages delivery) |
| **Ordering** | Guaranteed per partition | Guaranteed per queue (usually) |
| **Ideal Use Case** | Stream processing, Event Sourcing, Logging | Task queues, Complex routing, Request/Reply |

# Microservices

## Distributed Logging

In a microservices architecture, each service runs independently and often across different hosts or containers. Logs are generated in many places, so centralized visibility is essential for debugging, monitoring, auditing, and performance analysis.

Distributed logging enables:

* End to end request tracing across services
* Faster root cause analysis
* Real time monitoring and alerting
* Historical trend analysis
* Compliance and auditing

---

### Log Consolidation

Log consolidation collects logs from multiple services and centralizes them into a single system for storage and analysis.

This works well in:

* Heterogeneous environments where services use different languages and frameworks
* Hybrid deployments with VMs, containers, and serverless
* Systems where logs are batch processed or near real time

**Typical Flow**

1. Each microservice writes logs locally to stdout or to a file.
2. A log forwarder such as:

   * Logstash
   * Splunk
   * Loggly
     collects logs.
3. Logs are centralized into a unified index or storage system.
4. Analysis, dashboards, and alerts are built on top.

**Best Practices**

* Prefer structured logging such as JSON rather than plain text.
* Avoid writing directly to shared files from multiple services.
* Use stdout in containerized environments such as Kubernetes.
* Apply consistent log schemas across services.
* Set retention policies to control storage cost.

**Pros**

* Simpler operational model
* Easy to integrate with existing tools
* Good for post processing and reporting

**Cons**

* Less suitable for real time streaming analytics
* Can introduce delay depending on batching
* Central system can become a bottleneck

---

### Log Streaming

Log streaming treats logs as real time event streams. Instead of simply collecting logs into a central store, logs are published to a streaming platform and processed by subscribers.

This is common in event driven architectures.

**Typical Flow**

1. Microservices publish logs as events.
2. Logs are pushed into a distributed streaming system such as:

   * Apache Kafka
   * MapR Streams
3. Consumers subscribe to specific topics.
4. Subscribers:

   * Consolidate logs
   * Perform filtering
   * Trigger alerts
   * Run ad hoc analysis

**Advantages**

* Real time processing
* Scalable and fault tolerant
* Enables advanced analytics such as:

  * Error trend detection
  * Anomaly detection
  * Real time alerting
* Easy to plug and unplug consumers without impacting producers

**Best Practices**

* Separate topics by log level or domain if needed.
* Use partitioning strategies to preserve ordering where required.
* Ensure idempotent consumers to avoid duplicate processing.
* Monitor lag and throughput.
* Apply schema management using tools such as Confluent Schema Registry.

---

### Core Requirements for Distributed Logging

**1. Request Context ID**

Every request flowing through distributed services must carry a unique identifier.

Purpose:

* Trace a single request across services
* Correlate logs across boundaries
* Support debugging and auditing

Best practices:

* Generate at the edge, typically in an API gateway or first entry service.
* Propagate via headers such as X Request ID.
* Automatically inject into every log entry.

If using an API gateway, extract or generate the context ID there and propagate downstream.

---

**2. Context ID Hierarchy**

Sometimes multiple identifiers exist. A fallback hierarchy ensures meaningful traceability.

Example priority:

1. Customer ID
2. Account ID
3. Account number
4. Transaction ID
5. Generated request ID

Guidelines:

* Define a clear precedence order.
* Document it across teams.
* Never override a higher level identifier once set.
* Avoid mixing business identifiers with purely technical trace IDs unless clearly separated.

Industry best practice is to distinguish between:

* Correlation ID for business context
* Trace ID and Span ID for distributed tracing systems such as Jaeger or Zipkin.

---

**3. Context ID Extraction**

Use middleware or interceptors to ensure consistent extraction and injection.

Implementation pattern:

* HTTP interceptor extracts header.
* If missing, generate a new ID.
* Store in request scoped context.
* Ensure logging framework automatically reads from this context.

In frameworks:

* Java: Servlet filters or Spring interceptors
* Node.js: Middleware
* .NET: Delegating handlers

Critical rule: No service should log without a context ID.

---

**4. Context ID Consistency in Schema**

Logs must include the ID as a first class field, not embedded inside message strings.

Example JSON structure:
```js
{
"timestamp": "...",
"service": "payment-service",
"level": "ERROR",
"traceId": "...",
"customerId": "...",
"message": "Payment authorization failed"
}
```

Best practices:

* Enforce via logging wrapper.
* Validate schema at ingestion time.
* Reject malformed logs where possible.
* Avoid free form logging in production systems.

---

**5. Custom Logging API Wrapper**

A logging wrapper enforces consistency and reduces fragmentation across teams.

**Why It Matters**

Without a wrapper:

* Teams use different formats
* Fields are inconsistent
* Filtering logic is duplicated

With a wrapper:

* Standard schema
* Standard levels
* Automatic context injection
* Centralized filtering logic

**Advanced Routing**

You can route logs differently by severity:

* ERROR and FATAL to Apache Kafka
* DEBUG to Logstash
* INFO to RabbitMQ

This allows:

* High priority logs to be streamed in real time
* Lower priority logs to be batch processed
* Flexible evolution without modifying each service

Best practice:

* Keep the wrapper minimal.
* Do not hide underlying logging framework capabilities.
* Ensure async logging to avoid blocking application threads.
* Provide feature flags to dynamically change log levels.

---

### Additional Industry Best Practices

**Structured Logging Only**

Avoid plain text logs in distributed systems. Structured logs enable:

* Fast indexing
* Efficient filtering
* Better analytics
* Reduced parsing complexity

---

**Log Levels Discipline**

Define clear semantics:

* ERROR: Business failure or system failure
* WARN: Unexpected but recoverable
* INFO: Important lifecycle events
* DEBUG: Detailed diagnostic
* TRACE: Deep internal diagnostics

Avoid:

* Logging exceptions as INFO
* Logging sensitive information such as passwords or tokens

---

**Security and Compliance**

* Mask PII fields
* Encrypt logs at rest
* Enforce RBAC on log platforms
* Define retention policies

---

**Observability Integration**

Logging should complement:

* Metrics
* Distributed tracing

Modern observability platforms integrate all three pillars.

---

**Resilience**

* Logging failures must never crash services.
* Use non blocking appenders.
* Provide fallback mechanisms.
* Monitor logging pipeline health.


# Soft Skills



## The Knowledge Pyramid: Developers vs. Architects

Continously analyze technology and industry trends and keep current with the latest trends.

**The Three Tiers of Knowledge**
*   **Top:** Things You Know (Requires constant time investment to maintain).
*   **Middle:** Things You Know You Don’t Know.
*   **Bottom:** Things You Don’t Know You Don’t Know.

![](https://nealford.com/images/mark-pyramid-pt1.png)

**Developer Focus: Technical Depth**
*   Early career focuses on expanding the **top tier** to build hands-on experience and expertise.
*   The size of the top tier represents a developer's **technical depth**.

**Architect Focus: Technical Breadth**

> Technical Breadth: Stuff you know (technical depth) + stuff you know you don't know.

*   **Breadth > Depth:** An architect's value lies in understanding a wide variety of technologies to solve problems, rather than having deep expertise in just one.
*   It is more beneficial to know five potential solutions to a problem than to be a singular expert in one.
*   **Key Strategy:** Architects must sacrifice maintaining some of their deep, hard-won expertise and use that time to expand their **middle tier**, effectively broadening their technological portfolio to better match capabilities to constraints.


Few places you can expand your breadth knowledge
- InfoQ
- ThoughtWorks - Technology Rader
- DZone

---

**20 Minutes rule**

- Every morning, at the start of the day focus on expanding this area